## The Transformer Architecture +

### Performance Tuning:
 - **Batch Size**: This is how many sequences the GPU/CPU calculates in parallel. Instead of calculating 1 sequence of 64 words, we calculate 64 sequences of 64 words at once. It uses the hardware 64x better.
 - **Device**: If you have a Mac ($M1/M2/M3$), we use **mps**. If you have an NVIDIA GPU, we use **cuda**. This moves the math from the slow "General Purpose" CPU to the fast "Matrix Specialist" GPU.

In [11]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# --- Hyperparameters ---
batch_size = 64      # How many independent sequences will we process in parallel?
block_size = 128     # What is the maximum context length for predictions?
max_iters = 10000     # How many times to "nudge" the weights
eval_interval = 300  # How often to check if the model is getting smarter
learning_rate = 3e-4 # How big of a step to take during optimization
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
n_embd = 128         # Dimension of our word (character) coordinates
n_head = 4           # How many parallel "Attention Heads"
n_layer = 4          # How many Transformer Blocks to stack
# -----------------------



In [12]:
import requests

# 1. DATA: Load and Tokenize

with open('./data/gpt/kobzar.txt', 'r', encoding='utf-8') as file:
    text = file.read()

print(f"Loaded document from \"Кобзар\" source")
print(f"Total characters: {len(text):,}")


chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) 
train_data = data[:n]

def get_batch():
    # Pick random starting points in the text
    ix = torch.randint(len(train_data) - block_size, (batch_size,))
    x = torch.stack([train_data[i:i+block_size] for i in ix])
    y = torch.stack([train_data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)



Loaded document from "Кобзар" source
Total characters: 463,514


In [13]:
# 2. ARCHITECTURE: The explicit pieces
class Head(nn.Module):
    """ One head of self-attention """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,head_size)
        q = self.query(x) # (B,T,head_size)
        # Compute alignment scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # Causal mask
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        # Perform the weighted aggregation of the values
        v = self.value(x) # (B,T,head_size)
        out = wei @ v # (B, T, head_size)
        return out

class MultiHeadAttention(nn.Module):
    """ Multiple heads of self-attention in parallel """
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out

class FeedForward(nn.Module):
    """ The non-linear 'Brain' of the block """
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd), # Projecting back
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ One complete Transformer Block: Communication then Computation """
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # x + ... is the Residual/Skip connection
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class NanoGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # Final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # Combine what the word is and where it is
        x = self.blocks(x) # Go through the stacked blocks
        x = self.ln_f(x)
        logits = self.lm_head(x) # (B,T,vocab_size)

        loss = None
        if targets is not None:
            # CrossEntropy expects a specific shape: (Batch*Time, Vocab)
            loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:] # Crop context to block_size
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] # Focus only on the last time step
            probs = F.softmax(logits, dim=-1) # Probabilities
            idx_next = torch.multinomial(probs, num_samples=1) # Sample next character
            idx = torch.cat((idx, idx_next), dim=1) # Append to sequence
        return idx



In [14]:
# 3. TRAINING
model = NanoGPT().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    xb, yb = get_batch()
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward() # This is the "Backward" step for the whole system
    optimizer.step()

    if iter % eval_interval == 0:
        print(f"Step {iter}: Loss {loss.item():.4f}")



Step 0: Loss 4.9646
Step 300: Loss 2.6443
Step 600: Loss 2.5049
Step 900: Loss 2.4066
Step 1200: Loss 2.3080
Step 1500: Loss 2.2174
Step 1800: Loss 2.1937
Step 2100: Loss 2.0880
Step 2400: Loss 1.9857
Step 2700: Loss 1.9471
Step 3000: Loss 1.8819
Step 3300: Loss 1.8630
Step 3600: Loss 1.8340
Step 3900: Loss 1.7748
Step 4200: Loss 1.7454
Step 4500: Loss 1.7210
Step 4800: Loss 1.6934
Step 5100: Loss 1.7055
Step 5400: Loss 1.6962
Step 5700: Loss 1.7055
Step 6000: Loss 1.6740
Step 6300: Loss 1.6175
Step 6600: Loss 1.6374
Step 6900: Loss 1.6000
Step 7200: Loss 1.5621
Step 7500: Loss 1.5821
Step 7800: Loss 1.5826
Step 8100: Loss 1.5329
Step 8400: Loss 1.5453
Step 8700: Loss 1.5436
Step 9000: Loss 1.5587
Step 9300: Loss 1.5202
Step 9600: Loss 1.4945
Step 9900: Loss 1.4756


In [15]:
# 4. TEST: Generate 500 characters
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))


Босфоркилась та всесвля,
Та ні амотрись комундиком.
Так он, й одну кишенку,
Розбити з хати б намисніє;
Докутеся постоліли.

І заспіваласяі Окіна лихо;
На Мене гай палатах,
І страва й барвінок
Що в більше ніч йдучи.
Один моє!
Я по тьми долі зносило!
До дзвоненьки розказ
Та не дуже купі! За що!
Хто її мліте лину,
Горе твороноу,
Той помагали рано
На сім, що й несе божий,
Покуті в римені!
Блукаєш?" — літа!
У неволі народила!
Аж слав мати. За могилу
Ніже діночку і не зновами.
І не виталася душа,
У за


In [16]:
import ipywidgets as widgets
from IPython.display import display

# 5. INTERACTIVE COMPLETION INTERFACE
print("Building interactive widget interface...")

# Create widgets
prompt_input = widgets.Textarea(
    value='ой помру, тож поховайте...',
    placeholder='як би ви почали вірш...',
    description='Prompt:',
    layout=widgets.Layout(width='100%', height='120px')
)

tokens_slider = widgets.IntSlider(
    value=500,
    min=50,
    max=1500,
    step=50,
    description='Tokens to Gen:',
    layout=widgets.Layout(width='60%')
)

generate_btn = widgets.Button(
    description='Generate Completion',
    button_style='success',
    icon='pencil',
    layout=widgets.Layout(width='220px')
)

output_area = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='15px', margin='10px 0', background_color='#f9f9f9'))

def on_generate_clicked(b):
    output_area.clear_output()
    generate_btn.disabled = True
    generate_btn.description = 'Generating...'
    
    with output_area:
        print("Generating text, please wait...")
        
    try:
        prompt = prompt_input.value
        max_tokens = tokens_slider.value
        
        # Safe encoding: filter out any characters that aren't in the vocabulary
        encoded = [stoi[c] for c in prompt if c in stoi]
        if not encoded:
            encoded = [0] # Fallback if empty or all characters invalid
            
        context_tensor = torch.tensor([encoded], dtype=torch.long, device=device)
        
        model.eval()
        with torch.no_grad():
            # Generate text starting from the prompt context
            generated_tokens = model.generate(context_tensor, max_new_tokens=max_tokens)[0].tolist()
            
        completion_text = decode(generated_tokens)
        
        output_area.clear_output()
        with output_area:
            print(completion_text)
            
    except Exception as e:
        output_area.clear_output()
        with output_area:
            print(f"An error occurred during generation: {e}")
            
    finally:
        generate_btn.disabled = False
        generate_btn.description = 'Generate Completion'

generate_btn.on_click(on_generate_clicked)

# Display the widget layout
title_html = widgets.HTML("<h3>🎭 Shakespeare GPT Chat & Completion Interface 🎭</h3><p>Enter a prompt prefix, choose length, and generate a continuation.</p>")
controls = widgets.VBox([prompt_input, tokens_slider, generate_btn])
display(widgets.VBox([title_html, controls, output_area]))

Building interactive widget interface...
